# register-buffer — faded example 2: Implement init So Buffer Appears in state_dict But Not parameters (Faded)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-buffer`. Running the beacon reports progress on the `PyTorch: register_buffer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: register_buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-buffer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-buffer"
DD_SUBTOPIC = "PyTorch: register_buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The key distinction between `nn.Parameter` and `register_buffer` is optimizer visibility. Parameters flow into `.parameters()` and receive gradient updates. Buffers flow into `.buffers()` and `state_dict()` but are invisible to optimizers. A common mistake is using plain attribute assignment (`self.x = tensor`) which puts the tensor in neither `.parameters()` nor `state_dict()`.

## Faded exercise 2

A `NormStats` module needs a trainable `gamma` parameter and a non-trainable `running_var` buffer. The module skeleton shows where to add each. Your task is to **implement `register_buffer` for `running_var`**.

The `gamma` parameter registration is already shown. Add the buffer call for `running_var` (initialized to `torch.ones(features)`).

**Fill in:** Register running_var as a buffer initialized to ones of shape (features,) using self.register_buffer.

In [ ]:
import torch as t
import torch.nn as nn

class NormStats(nn.Module):
    def __init__(self, features: int):
        super().__init__()
        self.gamma = nn.Parameter(t.ones(features))
        raise NotImplementedError()  # TODO: Register running_var as a buffer initialized to ones of shape (features,) using self.register_buffer.

    def forward(self, x: t.Tensor) -> t.Tensor:
        return x / (self.running_var.sqrt() + 1e-5) * self.gamma


def _test():
    import torch as t
    m = NormStats(features=3)
    param_names  = [n for n, _ in m.named_parameters()]
    buffer_names = [n for n, _ in m.named_buffers()]
    sd_keys = list(m.state_dict().keys())
    assert 'running_var' in buffer_names, 'running_var should be a buffer'
    assert 'running_var' not in param_names, 'running_var should NOT be a parameter'
    assert 'running_var' in sd_keys, 'running_var should appear in state_dict'
    assert 'gamma' in param_names, 'gamma should be a parameter'
    assert not m.running_var.requires_grad
    assert t.allclose(m.running_var, t.ones(3))


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class NormStats(nn.Module):
    def __init__(self, features: int):
        super().__init__()
        self.gamma = nn.Parameter(t.ones(features))
        self.register_buffer('running_var', t.ones(features))

    def forward(self, x: t.Tensor) -> t.Tensor:
        return x / (self.running_var.sqrt() + 1e-5) * self.gamma
```
</details>